# Planning and Learning with Tabular Models

## Chapter Overview

This chapter develops a unified view of reinforcement learning methods that either require a model of the environment or can be used without a model.

- Methods that require a model include dynamic programming and heuristic search.
- Methods that can be used without a model include Monte Carlo and temporal-difference methods.
- These are respectively called **model-based** and **model-free** reinforcement learning methods.

Model-based methods rely on **planning** as their primary component, whereas model-free methods primarily rely on **learning**.

Although these two kinds of methods differ, they also have important similarities. The heart of both is the computation of value functions. Both look ahead to future events, compute a backed-up value, and use that backed-up value as an update target for an approximate value function.

Earlier Monte Carlo and temporal-difference methods were presented as distinct alternatives, then unified by n-step methods. This chapter similarly integrates model-based and model-free methods and explores how far they can be intermixed.

## 8.1 Models and Planning

A **model** of the environment is anything an agent can use to predict how the environment will respond to its actions. Given a state and an action, a model predicts the resultant next state and next reward.

If the model is stochastic, there are several possible next states and next rewards, each with some probability of occurring.

- A **distribution model** describes all possibilities and their probabilities.
- A **sample model** produces one possible transition, sampled according to the probabilities.

A distribution model is stronger than a sample model because it can always be used to produce samples. For example, a distribution model of a dozen dice gives all possible sums and their probabilities, and a sample model gives one sum sampled according to that probability distribution. Some sample models can be converted into distribution models by repeated queries, but this is possible only when the model is a sample model of a small finite set of possibilities.

Models can be used to mimic or simulate experience. Given a starting state and action, a sample model produces one transition; a distribution model generates all possible transitions weighted by their probabilities. In either case, the model is used to produce **simulated experience**.

**Planning** means any computational process that takes a model as input and produces or improves a policy for interacting with the modeled environment:

$$\text{model} \longrightarrow \text{planning} \longrightarrow \text{policy}$$

There are two distinct approaches to planning. **State-space planning** searches through the state space for an optimal policy or an optimal path to a goal. Actions cause transitions from state to state, and value functions are computed over states. This is the approach taken by dynamic programming and heuristic search. **Plan-space planning** searches through the space of plans; planning operators transform one plan into another. Plan-space methods are difficult to apply efficiently to the stochastic sequential decision problems that are the focus in reinforcement learning, and are not considered further.

All state-space planning methods share a common structure: they compute value functions as an intermediate step toward improving the policy. They do this by applying backups to simulated experience:

$$\text{model} \longrightarrow \text{simulated experience} \longrightarrow \text{backups} \longrightarrow \text{values} \longrightarrow \text{policy}$$

Dynamic programming clearly fits this structure. It sweeps through the space of states, generates the complete distribution of possible transitions from each state, and then uses that distribution to compute a backed-up value as an update target for the state's estimated value.

Other state-space planning methods differ in the kinds of updates they use, the order in which they do them, and how long the backed-up information is retained. The key point is that planning and learning share the same structure. Planning uses simulated experience generated by a model, whereas learning uses real experience generated by the environment. Because learning methods require only experience as input, they can be applied to simulated experience as well as to real experience.

### Random-Sample One-Step Tabular Q-Planning

Loop forever:

1. Select a state, $S \in \mathcal{S}$, and an action, $A \in \mathcal{A}(S)$, at random.
2. Send $S, A$ to a sample model, and obtain a sample next reward, $R$, and a sample next state, $S'$.
3. Apply one-step tabular Q-learning to $S, A, R, S'$:

$$Q(S,A) \leftarrow Q(S,A) + \alpha \left[R + \gamma \max_a Q(S',a) - Q(S,A)\right]$$

This method learns from a sample model and does not interact with the real environment. If each state-action pair is selected an infinite number of times in Step 1, and $\alpha$ decreases appropriately over time, it converges to the optimal policy for the real environment under the same conditions as one-step tabular Q-learning.

A second theme is the benefit of planning in small, incremental steps. This makes planning interruptible and resumable with little wasted computation, which is a key requirement for efficiently intermixing planning with acting and learning. Planning in very small steps may be inefficient in pure planning problems if the problem is too large to be solved exactly.

## 8.2 Dyna: Integrated Planning, Acting, and Learning

When planning is done online, while interacting with the environment, new information from interaction may change the model and thereby interact with planning. Planning may also be customized to the current state or to decisions currently under consideration or expected in the near future.

Within a planning agent, real experience has at least two roles:

- It can improve the model, called **model-learning**.
- It can directly improve the value function and policy, called **direct reinforcement learning**.

The Dyna architecture integrates these relationships. Direct reinforcement learning uses real experience to improve values and the policy. Model-learning uses real experience to improve the model. Planning uses the model to generate simulated experience, and backups from simulated experience improve values and the policy.

$$\text{real experience} \longrightarrow \text{direct RL update} \longrightarrow \text{policy/value functions}$$

$$\text{real experience} \longrightarrow \text{model learning} \longrightarrow \text{model} \longrightarrow \text{simulated experience} \longrightarrow \text{planning update} \longrightarrow \text{policy/value functions}$$

The process that selects starting states and actions for simulated experience is called **search control**. In Dyna-Q, search control selects state-action pairs randomly from those that have previously been experienced.

Dyna-Q includes planning, acting, model-learning, and direct reinforcement learning. Its planning method is random-sample one-step tabular Q-planning, its direct reinforcement learning method is one-step tabular Q-learning, and its model-learning method assumes a deterministic environment. After each transition, the model records in its table entry for $S,A$ that the predicted next reward and next state are $R,S'$.

During planning, the Q-planning algorithm samples only from state-action pairs that have previously been experienced, so the model is queried only with pairs for which it has information.

### Tabular Dyna-Q

Initialize $Q(s,a)$ and $Model(s,a)$ for all $s \in \mathcal{S}$ and $a \in \mathcal{A}(s)$.

Loop forever:

1. $S \leftarrow$ current nonterminal state.
2. $A \leftarrow \varepsilon$-greedy$(S,Q)$.
3. Take action $A$; observe resultant reward $R$ and state $S'$.
4. $Q(S,A) \leftarrow Q(S,A) + \alpha\left[R + \gamma \max_a Q(S',a) - Q(S,A)\right]$.
5. $Model(S,A) \leftarrow R,S'$ assuming a deterministic environment.
6. Repeat $n$ times:
   1. $S \leftarrow$ random previously observed state.
   2. $A \leftarrow$ random action previously taken in $S$.
   3. $R,S' \leftarrow Model(S,A)$.
   4. $Q(S,A) \leftarrow Q(S,A) + \alpha\left[R + \gamma \max_a Q(S',a) - Q(S,A)\right]$.

Conceptually, planning, acting, model-learning, and direct reinforcement learning occur simultaneously and in parallel. In Dyna-Q they occur sequentially by convention: acting, model-learning, direct reinforcement learning, then planning. The acting, model-learning, and direct reinforcement learning processes are assumed to consume only a fraction of the time, leaving time after each action for $n$ iterations of the Q-planning process.

### Dyna Maze

In the simple maze task, there are four actions: up, down, right, and left. Each action deterministically takes the agent to the corresponding neighboring state, except when movement is blocked by an obstacle or the edge of the maze, in which case the agent remains where it is. Reward is zero on all transitions, except for a reward of $+1$ after transition into the goal state. After reaching the goal state, the agent returns to the start state to begin a new episode. The task is discounted with $\gamma = 0.95$.

In the reported Dyna-Q experiments, the initial action values were zero, the agents were $\varepsilon$-greedy with $\varepsilon = 0.1$, and the exploration parameter was $\alpha = 0.1$. Ties in greedy action selection were broken randomly. Agents differed in the number of planning steps, $n$, performed per real step.

The $n = 0$ agent is a nonplanning agent that uses only direct reinforcement learning, which is one-step tabular Q-learning. For this maze it learned slowly. With $n = 5$, the task was solved much faster; with $n = 50$, it was solved faster still.

The planning agents found the solution faster because, after the first episode, planning had already built useful values from the model. Without planning, each episode added only one step to the policy. With planning, the model learned during the first episode was used to extend the greedy policy farther through simulated experience.

In Dyna-Q, learning and planning are accomplished by exactly the same algorithm operating on real experience for learning and simulated experience for planning. Planning proceeds incrementally, so it can be interrupted or redirected at any time.

## 8.3 When the Model Is Wrong

A model can be incorrect because the environment is stochastic and only a limited number of samples have been observed, because the model was learned using function approximation that generalized imperfectly, or because the environment has changed and its new behavior has not yet been observed. When the model is incorrect, planning is likely to compute a suboptimal policy.

In some cases, a suboptimal policy quickly leads to discovery and correction of the modeling error. This tends to happen when the model is **optimistic**, meaning it predicts greater reward or better state transitions than are actually possible. The planned policy then tries to exploit those opportunities and discovers that they do not exist.

### Blocking Maze

In the blocking maze, a short path from start to goal is available initially. After 1000 time steps, the short path is blocked and a longer path is opened. Both Dyna agents find the shortcut within 1000 steps. When the environment changes, the graphs become flat for a period because the agents obtain no reward while wandering around behind the barrier. After a while, they find the new opening and the new optimal behavior.

Greater difficulties arise when the environment changes to become better than it was before and the formerly correct policy does not reveal the improvement. In these cases, the modeling error may not be detected for a long time, if ever.

This is another version of the conflict between exploration and exploitation. In a planning context, **exploration** means trying actions that improve the model, whereas **exploitation** means behaving optimally given the current model. The agent should explore to find changes in the environment, but not so much that performance is greatly degraded. Simple heuristics are often effective.

### Dyna-Q+

The Dyna-Q+ agent that solved the shortcut maze uses one such heuristic. For each state-action pair, it keeps track of how many time steps have elapsed since the pair was last tried in a real interaction with the environment. The more time has elapsed, the greater the presumed chance that the dynamics of the pair have changed and that the model is incorrect.

To encourage behavior that tests long-untried actions, Dyna-Q+ gives a special **bonus reward** on simulated experiences involving these actions. If the modeled reward for a transition is $r$, and the transition has not been tried in $\tau$ time steps, then planning updates are done as if the transition produced reward

$$r + \kappa\sqrt{\tau}$$

for some small $\kappa$.

This encourages the agent to keep testing all accessible state transitions and to find long sequences of actions to carry out such tests. In the shortcut maze, this kind of computational curiosity is worth the extra exploration.

## 8.4 Prioritized Sweeping

In the Dyna agents described so far, simulated transitions start from state-action pairs selected uniformly at random from all previously experienced pairs. Uniform selection is usually not best. Planning can be more efficient if simulated transitions and updates focus on particular state-action pairs.

If one state changes in value, the useful updates are often not the arbitrary updates that might be selected uniformly. The useful updates are backups of predecessor state-action pairs that lead into the changed state. If those predecessor values change, their own predecessors may then become useful to update. This suggests working backward from states whose values have changed.

The value of a state changes when it is updated. The update's effect can be measured by the size of the change. Prioritized sweeping keeps a queue of state-action pairs whose estimated values would change nontrivially if updated, prioritized by the size of the change. When the pair in the queue is updated, the effect on its predecessor pairs is computed. If a predecessor's expected backup changes by more than a small threshold, the predecessor is inserted into the queue with its new priority.

### Prioritized Sweeping for a Deterministic Environment

Initialize $Q(s,a)$, $Model(s,a)$, for all $s,a$, and $PQueue$ to empty.

Loop forever:

1. $S \leftarrow$ current nonterminal state.
2. $A \leftarrow policy(S,Q)$.
3. Take action $A$; observe resultant reward $R$ and state $S'$.
4. $Model(S,A) \leftarrow R,S'$.
5. $P \leftarrow \left|R + \gamma \max_a Q(S',a) - Q(S,A)\right|$.
6. If $P > \theta$, then insert $S,A$ into $PQueue$ with priority $P$.
7. Loop repeat $n$ times, while $PQueue$ is not empty:
   1. $S,A \leftarrow first(PQueue)$.
   2. $R,S' \leftarrow Model(S,A)$.
   3. $Q(S,A) \leftarrow Q(S,A) + \alpha\left[R + \gamma \max_a Q(S',a) - Q(S,A)\right]$.
   4. Loop for all $\bar{S},\bar{A}$ predicted to lead to $S$:
      1. $\bar{R} \leftarrow$ predicted reward for $\bar{S},\bar{A},S$.
      2. $P \leftarrow \left|\bar{R} + \gamma \max_a Q(S,a) - Q(\bar{S},\bar{A})\right|$.
      3. If $P > \theta$, then insert $\bar{S},\bar{A}$ into $PQueue$ with priority $P$.

Extensions of prioritized sweeping to stochastic environments are straightforward. The model keeps counts of how many times each state-action pair has been experienced and what the next states were. Each pair is updated not with a sample update, but with an expected update that takes into account all possible next states and their probabilities.

Prioritized sweeping is one way of distributing computations to improve planning efficiency, but it is probably not the best way. One limitation is that it uses expected updates. In stochastic environments, many wasteful expected updates may be spent on low-probability transitions. Sample updates can in many cases get closer to the true value function with less computation, despite the variance introduced by sampling.

Sample updates break the overall backup computation into smaller pieces, each corresponding to an individual transition. The aim of **small backups** is to focus on the pieces that have the largest impact. These are updates along a single transition, like a sample update, but based on the probability of the transition without sampling, as in an expected update. Selecting the order in which small updates are done can greatly improve planning efficiency.

State-space planning can be viewed as sequences of value updates. Planning methods vary only in the type of update, expected or sample, large or small, and in the order in which updates are done. Prioritized sweeping emphasizes backward focusing, but this is only one strategy. Forward focusing can focus backups on states according to how easily they can be reached from the states visited frequently under the current policy.

## 8.5 Expected vs. Sample Updates

Updates differ along several dimensions. They may update state values or action values. They may estimate the value for the optimal policy or for an arbitrary given policy. They may use expected updates or sample updates. They may also differ in whether they update from entire distributions or from single sample backups.

Expected updates compute the exact backed-up value from all possible next states, rewards, and their probabilities. For an action-value estimate $Q$, a model with estimated dynamics $\hat{p}(s',r \mid s,a)$, and an action $a^*$ that maximizes the current action values at the next state, the expected update for a state-action pair is

$$Q(s,a) \leftarrow \sum_{s',r} \hat{p}(s',r \mid s,a)\left[r + \gamma \max_{a'} Q(s',a')\right].$$

The corresponding sample update samples a next state and reward, $S'$ and $R$, from the model and uses the Q-learning-like update

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[R + \gamma \max_a Q(S',a) - Q(s,a)\right].$$

If the same sample update is presented many times, the estimate converges to the expected update. The difference between expected and sample updates is significant when the environment is stochastic. If the environment is deterministic, the expected and sample updates are identical except that the expected update requires more computation.

The expected update is limited by the correctness of $Q$ at successor states. The sample update is limited by that same source of error and by sampling error. In this sense, an expected update gives a better estimate than a sample update because it is uncorrupted by sampling error. But expected updates also require more computation.

With $b$ possible next states, one expected update requires about $b$ times as much computation as one sample update. If one expected update and $b$ sample updates use comparable computation, then the comparison depends on how quickly the sample updates reduce sampling error. Sample updates reduce sampling error according to $\sqrt{b}$, where $b$ is the number of sample updates performed, assuming the sample averages have independent errors.

The advantage of sample updates is probably underestimated when all successor states are equally likely. In real problems, transition probabilities are usually skewed. Some transitions may have high probability and dominate the expected update. Sample updates will naturally focus on the more likely successor states and may gain an advantage because the values backed up from those states are more accurate.

Thus, when there are many possible next states and a large stochastic branching factor, sample updates can be preferable to expected updates. They can be much cheaper computationally, and the backup values may be closer to the true value function even with sampling variance.

## 8.6 Trajectory Sampling

The classical way to distribute updates, from dynamic programming, is to perform sweeps through the entire state or state-action space, updating each state or state-action pair once per sweep. This is problematic on large tasks because there may not be enough time to complete even one sweep, and many states may be irrelevant because they are visited only under very poor policies or with very low probability.

Exhaustive sweeps implicitly give equal time to all parts of the state space rather than focusing where it is needed. They are often used with expected updates, but the distribution of updates and the type of update are separate ideas.

A second approach is to sample from the state or state-action space according to a distribution. Uniform sampling has some of the same problems as exhaustive sweeps. A more appealing choice is to distribute updates according to the **on-policy distribution**, the distribution observed when following the current policy. This is called **trajectory sampling**.

The on-policy distribution can be generated by interacting with the model: follow the current policy, select actions as the policy would, and use the model to generate transitions and rewards. This produces simulated trajectories and updates state-action pairs according to how often they occur under the current policy.

There is a tradeoff. Sampling according to the on-policy distribution may waste updates on parts of the state space that are already being visited. Uniform sampling may perform useful work on other states. But for large problems, where only a small subset of the state-action space is visited under the on-policy distribution, trajectory sampling can be much more efficient.

### Comparison Experiment

The comparison used randomly generated tasks with $1000$ and $10000$ states. From each state, two actions were possible. Each action led to one of $b$ next states, all equally likely, with a different random selection of $b$ states for each state-action pair. The branching factor $b$ was the same for all state-action pairs. On all transitions there was a $0.1$ probability of transition to the terminal state, ending the episode. The expected reward on each transition was selected from a Gaussian distribution with mean $0$ and variance $1$.

For branching factors $b=1$, $b=3$, and $b=10$, updates were distributed either uniformly or according to the on-policy distribution. The value shown was the true value of the start state under the greedy policy implied by the current action-value function $Q$.

For all cases with $b=1$, sampling according to the on-policy distribution learned faster than uniform sampling and also reached a better long-run result. In the larger problem, sampling according to the on-policy distribution was better early for all branching factors. With large branching factors, uniform sampling was better in the long run.

The advantage of on-policy sampling in the short run is that it focuses updates near the start state. If there are many states and a small branching factor, this effect can be large and long lasting. With a large branching factor, many states become successors of the start state, so focusing on the on-policy distribution may have less effect. These results suggest that sampling according to the on-policy distribution can be a great advantage for large problems, especially when only a small subset of the state-action space is visited under the on-policy distribution.

## 8.7 Real-time Dynamic Programming

**Real-time dynamic programming** (RTDP) is an on-policy trajectory-sampling version of the value-iteration algorithm of dynamic programming. Because it is closely related to conventional sweep-based policy iteration, RTDP shows that on-policy trajectory sampling can provide some of the advantages of dynamic programming.

RTDP updates the values of states visited in actual or simulated trajectories by means of expected tabular value-iteration updates. The update order is not an exhaustive sweep; it is dictated by the order in which states are visited in real or simulated trajectories.

This makes RTDP an example of an asynchronous dynamic programming algorithm. Asynchronous DP algorithms are not organized as systematic sweeps of the state set; they update state values in any order, as long as the values of all states required for convergence are updated.

### Relevant States

If trajectories can start only from a designated set of start states, then for a control problem one may need an optimal policy only for **relevant states**. These are states that can be reached from the start states under an optimal policy. States that cannot be reached by any optimal policy from the start states do not need optimal actions specified.

For a prediction problem with a fixed policy, the value function needs to be accurate only for states reached under that policy. For a control problem, the goal is an optimal policy rather than a value function that is optimal on all states.

### RTDP for Stochastic Shortest Path Problems

The convergence results for RTDP are for undiscounted episodic tasks for MDPs with absorbing goal states that generate zero rewards. At every step of a real or simulated trajectory, RTDP selects a greedy action, breaking ties randomly, and applies the expected value-iteration update operation to the current state. RTDP can also update values of other states at each step.

For these problems, if each episode begins in a state randomly chosen from the start states and ends at a goal state, then RTDP converges with probability one to a policy optimal on every start state if:

1. The initial value of every goal state is zero.
2. There exists at least one policy that guarantees a goal state will be reached with probability one from any start state.
3. All rewards for transitions from non-goal states are strictly negative.
4. All initial values are equal to, or greater than, their optimal values.

This is proved by showing that RTDP follows a constrained form of asynchronous DP with an initially optimistic value function.

### Why RTDP Can Be Useful Before Convergence

As the value function approaches the optimal value function, the policy used to generate trajectories approaches an optimal policy because it is always greedy with respect to the current value function. Thus, in contrast to conventional value iteration, RTDP can be used to obtain good policies quickly even though the value function has not yet converged.

RTDP is computationally efficient because it focuses updates on states that are relevant to the problem objective. This focus can greatly reduce computation compared with sweeps over the full state set.

## 8.8 Planning at Decision Time

Planning can be used in at least two ways. The first way is the kind considered so far in this chapter: planning gradually improves a policy or value function on the basis of simulated experience obtained from a model. Selecting an action is then a matter of comparing the current action values, or evaluating a mathematical expression in the approximate methods. Planning has already improved the table entries or function-approximation parameters needed to select actions for many states, including the current state $S_t$.

Used this way, planning is not focused on the current state. This is called **background planning**.

The second way is to begin and complete planning after encountering each new state $S_t$. The output is the selection of a single action $A_t$ in that next step. The action may be selected by comparing the values of model-predicted next states for each action, or by comparing the values of action-reward trajectories. Planning used this way can look deeper than one-step-ahead and evaluate choices leading to many different predicted state and reward trajectories.

Unlike background planning, this use of planning focuses on a particular state. This is called **decision-time planning**.

Decision-time planning is a way to use the current state and the current model to select the current action. The values and policy are specific to the current state and the action choices available there, so much of the computation may be discarded after the current action is selected. Later states may not be the same as the current one.

Decision-time planning is most useful when fast responses are not required. In some planning programs, computation for each move can take seconds or minutes. If low-latency action selection is the priority, background planning can compute a policy that can be rapidly applied to each newly encountered state.

## 8.9 Heuristic Search

Classical state-space planning methods in artificial intelligence are decision-time planning methods, collectively known as **heuristic search**. In heuristic search, each state has an estimate of how likely the agent is to reach the goal from that state. This value function is called a **heuristic evaluation function**.

From a current state, a large tree of possible continuations is considered. The approximate value function is applied to the leaf nodes and then backed up toward the current state at the root. The backing-up can be done with the same kinds of updates used throughout this chapter. Once the backed-up values are known, the best current action is selected, and the backed-up values are discarded.

In conventional heuristic search, the backed-up values are often not saved. In reinforcement learning, they can be saved by replacing the current value of each backed-up state with its backed-up value. If the same states are encountered again, the improved values can make later decision-time planning more efficient.

The distinction between heuristic search and the planning methods already considered is partly the distribution of updates. Heuristic search focuses updates on the current state and its likely successors. Earlier planning methods may distribute updates more broadly, such as across previously experienced state-action pairs or according to simulated trajectories.

This focus can be useful when the agent needs to choose the next action from the current state. The sequence of one-step updates backs up values from leaf nodes toward the root, and the ordering of these backups can be changed to emphasize parts of the search tree that matter for the current decision.

## 8.10 Rollout Algorithms

**Rollout algorithms** are decision-time planning algorithms based on Monte Carlo control applied to simulated trajectories that all begin at the current environment state. They estimate action values for a given policy by averaging returns of many simulated trajectories that start with each possible action and then follow the given policy. When the action-value estimates are considered accurate enough, the action with the highest estimated value is executed, and the process is repeated at the next state.

The given policy is called the **rollout policy**, and the process of estimating an action value by averaging many simulated trajectories is called **rolling out**. In some games, rollout trajectories can be generated using a fixed policy.

Unlike Monte Carlo control algorithms described earlier, rollout algorithms do not estimate a complete optimal action-value function $q_*$. Instead, they produce Monte Carlo estimates of action values only for the current state and for a given policy. As decision-time planning algorithms, rollout algorithms produce only the one action needed now and then discard the intermediate action-value estimates.

The policy improvement theorem applies. If the rollout policy is $\pi$ and $q_\pi(s,a)$ is known for each action available in the current state $s$, then selecting an action with maximal $q_\pi(s,a)$ gives a policy at least as good as $\pi$ in that state. The rollout algorithm estimates these values by averaging returns from simulated trajectories for each action. If enough trajectories are averaged, the selected action is expected to improve over the rollout policy.

Experience shows that rollout algorithms can improve performance. The better the rollout policy and the more accurate the value estimates, the better the policy produced by rollout is likely to be.

Rollout algorithms involve important tradeoffs. Better rollout policies usually require more time in simulated trajectories to obtain good value estimates. The computation time depends on how many actions must be evaluated, how many trajectories are needed for each action, and how many time steps each simulated trajectory needs to obtain useful sample returns. These factors must be balanced in each application.

Rollout algorithms take advantage of sample trajectories and sampling over exhaustive sweeps of dynamic programming. They estimate action values by averaging sampled trajectories, avoiding distribution models, and they act greedily with respect to estimated action values.

## 8.11 Monte Carlo Tree Search

**Monte Carlo Tree Search** (MCTS) is a decision-time planning algorithm. It is a rollout algorithm enhanced by a means for accumulating value estimates obtained from Monte Carlo simulations in order to direct simulations toward more highly rewarding trajectories.

MCTS is executed after each new state is encountered to select the agent's action for that state. Each execution is an iterative process that simulates many trajectories starting from the current state and running to a terminal state, or until discounting makes further reward negligible. MCTS successively focuses multiple simulations starting at the current state by extending the initial portions of trajectories that have received high evaluations from earlier simulations.

MCTS does not have to retain approximate value functions or policies from one action selection to the next, although many implementations retain selected action values likely to be useful for the next execution.

Two policies are used. The **tree policy** selects actions for states already represented in the tree. The **rollout policy** selects actions beyond the tree. The tree is grown by adding nodes for states reached from selected leaf nodes, and action values in the tree are updated from the simulated returns.

Each iteration of MCTS has four steps:

1. **Selection**: Starting at the root node, choose actions according to the tree policy until a leaf node is reached.
2. **Expansion**: On some iterations, expand the tree from the selected leaf node by adding one or more child nodes reached from the selected node by unexplored actions.
3. **Simulation**: From the selected node, or from one of its newly added child nodes, simulate a complete episode using the rollout policy.
4. **Backup**: Back up the return from the simulated episode to update, or initialize, action values attached to the edges of the tree traversed by the tree policy.

MCTS continues these four steps, starting each time at the tree's root node, until no more time is left or another computational resource is exhausted. Then the action from the root node with the highest estimated value is selected. The tree policy may use a mechanism that balances exploration and exploitation; after the environment transitions to a new state, the subtree from that state may be reused, and the rest of the tree is discarded.

Relative to reinforcement learning, MCTS is a decision-time planning algorithm based on Monte Carlo control applied to simulations that start from the root state. It improves action-value estimates by sample updates and grows a lookup table of a partial action-value function. It does not change the policy through global approximation of an action-value function; instead, it uses the sample experience to guide exploration.